In [0]:
# Silver Layer: Date and Time Transformations
# Author: Virendra Dilip Tambavekar
# HRM ID: 6217
# Domain: Cross-Domain Reference
# Source: bronze.date, bronze.time
# Target: silver.date, silver.time
# Description: Multi-format date/time parsing, quarantine invalid records,
#              derive calendar and time dimension fields.

In [0]:
#Importing required libraries
import logging
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType

#Initialize Logger
logger = logging.getLogger("BronzeToSilver_DateTime")
logger.setLevel(logging.INFO)

In [0]:
#Parameterization
dbutils.widgets.text("batch_id","1")

batch_id = dbutils.widgets.get("batch_id")
#schemas
bronze_schema = "charles_schwab_retailbrokerage_dev_team_lemma.bronze"
silver_schema = "charles_schwab_retailbrokerage_dev_team_lemma.silver"
quarantine_schema = "charles_schwab_retailbrokerage_dev_team_lemma.quarantine"

In [0]:
def process_date_dimension() -> dict:
    logger.info("Processing Date dimension (Bronze -> Silver)...")
    bronze_date_df = spark.table(f"{bronze_schema}.date").dropDuplicates(["record_id"])
    bronze_count = bronze_date_df.count()

    # 1. Normalization & Cascading Multi-Format Parse
    date_parsed_df = (
        bronze_date_df
        .withColumn("_trimmed_date", trim(col("source_date_string")))
        .withColumn("_title_date", initcap(col("_trimmed_date")))
        .withColumn(
            "_raw_date",
            coalesce(
                try_to_date(col("_trimmed_date"), "MM/dd/yyyy"), # CRM-01
                try_to_date(col("_trimmed_date"), "yyyy.MM.dd"), # ERP-SAP
                try_to_date(col("_title_date"), "dd-MMM-yy"),    # MF-LEGACY
                try_to_date(col("_trimmed_date"), "MMMM d, yyyy"), # TRD-DESK
                try_to_date(col("_trimmed_date"), "MMMM dd, yyyy"),
                try_to_date(col("_trimmed_date"), "MMM d, yyyy"),
                try_to_date(col("_trimmed_date"), "MMM dd, yyyy")
            )
        )
        # Year correction ONLY for MF-LEGACY 2-digit years
        .withColumn(
            "DateValue",
            when(
                (year(col("_raw_date")) > 2020) & (trim(col("source_system_code")) == "MF-LEGACY"),
                add_months(col("_raw_date"), -1200)
            ).otherwise(col("_raw_date"))
        )
        .drop("_trimmed_date", "_title_date", "_raw_date")
    )

    # 2. Identify Failures, Duplicates, and Out-of-Bounds
    date_window = Window.partitionBy("DateValue").orderBy(col("_ingest_ts").desc())
    
    evaluated_df = (
        date_parsed_df
        .withColumn("_row_num", row_number().over(date_window))
        .withColumn(
            "_reject_reason",
            when(col("DateValue").isNull(), "PARSE_FAILED")
             .when(col("_row_num") > 1, "DUPLICATE")
             .when((col("DateValue") < "1950-01-01") | (col("DateValue") > "2020-12-31"), "OUT_OF_BOUNDS")
             .otherwise(lit(None))
        )
    )

    # 3. Route to Quarantine
    date_quarantine_all = (
        evaluated_df.filter(col("_reject_reason").isNotNull())
        .select("record_id", "source_date_string", "source_system_code", "_reject_reason")
        .withColumn("_quarantined_at", current_timestamp())
        .withColumn("_batch", lit(batch_id))
    )
    quarantine_count = date_quarantine_all.count()

    # 4. Route to Silver and Derive Calendar Dimension Fields
    silver_date_df = evaluated_df.filter(col("_reject_reason").isNull()).select(
        col("DateValue"),
        date_format(col("DateValue"), "MMMM d, yyyy").alias("DateDesc"),
        year(col("DateValue")).cast(IntegerType()).alias("CalendarYearID"),
        concat(lit("CY"), year(col("DateValue")).cast("string")).alias("CalendarYearDesc"),
        (year(col("DateValue")) * 10 + quarter(col("DateValue"))).cast(IntegerType()).alias("CalendarQtrID"),
        concat(year(col("DateValue")).cast("string"), lit(" Q"), quarter(col("DateValue")).cast("string")).alias("CalendarQtrDesc"),
        (year(col("DateValue")) * 100 + month(col("DateValue"))).cast(IntegerType()).alias("CalendarMonthID"),
        date_format(col("DateValue"), "MMMM yyyy").alias("CalendarMonthDesc"),
        (year(col("DateValue")) * 100 + weekofyear(col("DateValue"))).cast(IntegerType()).alias("CalendarWeekID"),
        concat(year(col("DateValue")).cast("string"), lit(" W"), lpad(weekofyear(col("DateValue")).cast("string"), 2, "0")).alias("CalendarWeekDesc"),
        when(dayofweek(col("DateValue")) == 1, 7).otherwise(dayofweek(col("DateValue")) - 1).cast(IntegerType()).alias("DayOfWeekNum"),
        date_format(col("DateValue"), "EEEE").alias("DayOfWeekDesc"),
        when(month(col("DateValue")) >= 7, year(col("DateValue")) + 1).otherwise(year(col("DateValue"))).cast(IntegerType()).alias("FiscalYearID"),
        when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))).alias("FiscalYearDesc"),
        when(month(col("DateValue")).between(7, 9), 1).when(month(col("DateValue")).between(10, 12), 2).when(month(col("DateValue")).between(1, 3), 3).otherwise(4).cast(IntegerType()).alias("FiscalQtrID"),
        when(month(col("DateValue")).between(7, 9), concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q1")))
         .when(month(col("DateValue")).between(10, 12), concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q2")))
         .when(month(col("DateValue")).between(1, 3), concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q3")))
         .otherwise(concat(when(month(col("DateValue")) >= 7, concat(lit("FY"), (year(col("DateValue")) + 1).cast("string"))).otherwise(concat(lit("FY"), year(col("DateValue")).cast("string"))), lit(" Q4"))).alias("FiscalQtrDesc"),
        lit(False).cast(BooleanType()).alias("HolidayFlag"),
        lit(batch_id).alias("_batch"),
        current_timestamp().alias("_load_ts")
    )
    silver_count = silver_date_df.count()

    # 5. Idempotent Writes
    silver_date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.date")
    date_quarantine_all.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{quarantine_schema}.date_quarantine")
    
    return {"Dataset": "Date", "Bronze_Count": bronze_count, "Quarantined_Count": quarantine_count, "Silver_Count": silver_count}

In [0]:
from functools import reduce as _reduce

def process_time_dimension() -> dict:
    logger.info("Processing Time dimension (Bronze -> Silver)...")
    bronze_time_df = spark.table(f"{bronze_schema}.time").dropDuplicates(["record_id"])
    bronze_count = bronze_time_df.count()

    has_extract = "extract_batch_id" in bronze_time_df.columns
    base_cols = ["record_id", "source_time_string", "source_system_code", "time_precision"]
    if has_extract: base_cols.append("extract_batch_id")
        
    common_cols = [*base_cols, "standard_time"]
    quarantine_parts = []

    def to_quarantine(df, reason_code):
        return (df.select(*common_cols)
                  .withColumn("_reject_reason", lit(reason_code))
                  .withColumn("_quarantined_at", current_timestamp())
                  .withColumn("_batch", lit(batch_id)))

    # 1. Parse and Normalize
    base = (
        bronze_time_df.select(*base_cols)
        .withColumn("clean_time", trim(col("source_time_string")))
        .withColumn("norm_time", regexp_replace(regexp_replace(col("clean_time"), r"(?i)a\.m\.", "AM"), r"(?i)p\.m\.", "PM"))
    )

    # 2. Quarantine UNKNOWN precision
    quarantine_parts.append(to_quarantine(base.filter(col("time_precision") == "UNKNOWN").withColumn("standard_time", lit(None).cast("string")), "UNKNOWN_PRECISION"))
    base = base.filter(col("time_precision") != "UNKNOWN")

    # 3. Expand MINUTE times & Convert numeric formats
    base = (
        base.withColumn(
            "expanded_time",
            when(col("time_precision") == "MINUTE",
                 when(col("norm_time").rlike(r"^\d{4}$"), concat(col("norm_time"), lit("00")))
                 .otherwise(regexp_replace(col("norm_time"), r"(?i)\s+(AM|PM)$", ":00 $1")))
            .otherwise(col("norm_time"))
        )
        .withColumn("formatted_numeric", regexp_replace(lpad(col("expanded_time"), 6, "0"), r"(\d{2})(\d{2})(\d{2})", "$1:$2:$3"))
        .withColumn("parsed_ampm", date_format(expr("try_to_timestamp(expanded_time, 'h:mm:ss a')"), "HH:mm:ss"))
    )

    # 4. Derive standard_time
    base = base.withColumn(
        "standard_time",
        coalesce(
            when(col("expanded_time").rlike(r"^\d{1,2}:\d{2}:\d{2}$"), col("expanded_time")),
            col("parsed_ampm"),
            when(col("expanded_time").rlike(r"^\d{6}$"), col("formatted_numeric"))
        )
    )
    base = base.withColumn("standard_time", when(col("standard_time").rlike(r"^\d:\d{2}:\d{2}$"), concat(lit("0"), col("standard_time"))).otherwise(col("standard_time")))

    # 5. Quarantine NULL_TIME and DUPLICATE_BATCH
    quarantine_parts.append(to_quarantine(base.filter(col("standard_time").isNull()), "NULL_TIME"))
    base_valid = base.filter(col("standard_time").isNotNull())
    
    if has_extract:
        quarantine_parts.append(to_quarantine(base_valid.filter(col("extract_batch_id").contains("DUP")), "DUPLICATE_BATCH"))
        base_valid = base_valid.filter(~col("extract_batch_id").contains("DUP"))

    # 6. Split SECOND / MINUTE, Expand MINUTE into 60 SECONDS
    second_df = base_valid.filter(col("time_precision") == "SECOND").select(*common_cols)
    minute_df = (
        base_valid.filter(col("time_precision") == "MINUTE").select(*common_cols)
        .withColumn("sec", explode(sequence(lit(0), lit(59))))
        .withColumn("standard_time", concat_ws(":", substring(col("standard_time"), 1, 5), lpad(col("sec").cast("string"), 2, "0")))
        .drop("sec")
    )

    # 7. Union, Deduplicate to exactly 86,400 seconds, and Build Final Silver Dimensions
    time_silver_base = second_df.union(minute_df).dropDuplicates(["standard_time"])

    silver_time_df = time_silver_base.select(
        col("standard_time").alias("TimeValue"),
        expr("CAST(substring(standard_time, 1, 2) AS INT)").alias("HourID"),
        concat(substring(col("standard_time"), 1, 2), lit(":00")).alias("HourDesc"),
        expr("CAST(substring(standard_time, 4, 2) AS INT)").alias("MinuteID"),
        substring(col("standard_time"), 1, 5).alias("MinuteDesc"),
        expr("CAST(substring(standard_time, 7, 2) AS INT)").alias("SecondID"),
        col("standard_time").alias("SecondDesc"),
        when((expr("CAST(substring(standard_time, 1, 2) AS INT)") >= 9) & (expr("CAST(substring(standard_time, 1, 2) AS INT)") < 16), lit(True)).otherwise(lit(False)).cast(BooleanType()).alias("MarketHoursFlag"),
        when((expr("CAST(substring(standard_time, 1, 2) AS INT)") >= 8) & (expr("CAST(substring(standard_time, 1, 2) AS INT)") < 17), lit(True)).otherwise(lit(False)).cast(BooleanType()).alias("OfficeHoursFlag"),
        lit(batch_id).alias("_batch"),
        current_timestamp().alias("_load_ts")
    )
    silver_count = silver_time_df.count()

    # Combine Quarantines and Write Idempotently
    time_quarantine_all = _reduce(lambda a, b: a.unionByName(b), quarantine_parts)
    quarantine_count = time_quarantine_all.count()

    silver_time_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.time")
    time_quarantine_all.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{quarantine_schema}.time_quarantine")
    
    # STANDARDIZED KEYS
    return {"Dataset": "Time", "Bronze_Count": bronze_count, "Quarantined_Count": quarantine_count, "Silver_Count": silver_count}

In [0]:
def main():
    logger.info("Starting Date and Time Silver Layer")

    try:
        date_metrics = process_date_dimension()
        time_metrics = process_time_dimension()

        logger.info("Silver Reconciliation & Audit Log")
        recon_df = spark.createDataFrame([date_metrics, time_metrics])
        display(recon_df)

        logger.info("Pipeline Completed")
    except Exception as e:
        logger.error(f"Error processing date and time silver layer: {e}")
        raise e

main()